# 05 - Segmentation Model (EfficientNet+UNet)

Train segmentation trên pseudo-label masks từ notebook 04.
Chia train/val/test từ pseudo-label dataset.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

# Auto-detect project root
notebook_dir = os.path.abspath('')
proj_root = os.path.dirname(notebook_dir) if os.path.basename(notebook_dir) == 'notebooks' else notebook_dir
sys.path.insert(0, proj_root)
sys.path.insert(0, os.path.join(proj_root, 'utils'))

import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import torchvision.transforms.functional as TF
import numpy as np
import cv2
import json
import random
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

from utils.models import EfficientNetUNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('proj_root:', proj_root)

## 1) Load pseudo-label dataset từ notebook 04

In [ ]:
# Đường dẫn từ notebook 04
IMAGE_DIR = os.path.join(proj_root, 'notebooks', 'data', 'processed_train')
MASK_DIR  = os.path.join(proj_root, 'notebooks', 'data', 'pseudo_labels_train')

# Fallback nếu không tìm thấy
if not os.path.exists(IMAGE_DIR):
    IMAGE_DIR = 'data/processed_train'
if not os.path.exists(MASK_DIR):
    MASK_DIR = 'data/pseudo_labels_train'

print('IMAGE_DIR:', IMAGE_DIR, '| exists:', os.path.exists(IMAGE_DIR))
print('MASK_DIR:', MASK_DIR, '| exists:', os.path.exists(MASK_DIR))

if os.path.exists(IMAGE_DIR):
    imgs_sample = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.png'))][:5]
    print('Images (5 đầu):', imgs_sample)
if os.path.exists(MASK_DIR):
    masks_sample = [f for f in os.listdir(MASK_DIR) if f.endswith('_pseudo.png')][:5]
    print('Masks (5 đầu):', masks_sample)

In [ ]:
class DurianSegDataset(Dataset):
    def __init__(self, image_dir, mask_dir, image_size=224, augment=False):
        self.image_dir = image_dir
        self.mask_dir  = mask_dir
        self.image_size = image_size
        self.augment   = augment

        self.samples = []
        for fname in os.listdir(mask_dir):
            if not fname.endswith('_pseudo.png'):
                continue
            # e.g. ALGAL_LEAF_SPOT_to_label_100.jpg_pseudo.png -> ALGAL_LEAF_SPOT_to_label_100.jpg
            img_name = fname.replace('_pseudo.png', '')
            img_path  = os.path.join(image_dir, img_name)
            mask_path = os.path.join(mask_dir, fname)
            if os.path.exists(img_path):
                self.samples.append((img_path, mask_path))

        print(f'[Dataset] Found {len(self.samples)} image-mask pairs')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        img  = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')

        img  = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        mask = mask.resize((self.image_size, self.image_size), Image.NEAREST)

        if self.augment:
            if random.random() > 0.5:
                img  = TF.hflip(img)
                mask = TF.hflip(mask)
            if random.random() > 0.5:
                img  = TF.vflip(img)
                mask = TF.vflip(mask)

        img  = TF.to_tensor(img)
        img  = TF.normalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        mask = TF.to_tensor(mask)
        mask = (mask > 0.5).float()

        return {
            'image': img,
            'label': mask,
            'img_path': img_path,
        }


full_dataset = DurianSegDataset(IMAGE_DIR, MASK_DIR, augment=False)

n = len(full_dataset)
n_train = int(0.7 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val

print(f'Split: train={n_train}, val={n_val}, test={n_test}')

# Fix seed để reproducible
generator = torch.Generator().manual_seed(42)
train_seg, val_seg, test_seg = random_split(full_dataset, [n_train, n_val, n_test], generator=generator)

# Bật augment cho train
train_seg.dataset.augment = True

train_loader = DataLoader(train_seg, batch_size=8, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_seg,   batch_size=8, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_seg,  batch_size=8, shuffle=False, num_workers=0)

batch = next(iter(train_loader))
print('image:', batch['image'].shape)
print('label:', batch['label'].shape)

## 2) Model & loss

In [ ]:
model = EfficientNetUNet(num_classes=1, pretrained=True).to(device)
criterion_bce = nn.BCEWithLogitsLoss()

def dice_loss(pred, target, smooth=1):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1 - dice

optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

## 3) Metrics

In [ ]:
def calculate_segmentation_metrics(pred, gt, smooth=1e-6):
    pred = pred.flatten().astype(bool)
    gt   = gt.flatten().astype(bool)

    tp = (pred & gt).sum()
    fp = (pred & ~gt).sum()
    fn = (~pred & gt).sum()

    precision = (tp + smooth) / (tp + fp + smooth)
    recall    = (tp + smooth) / (tp + fn + smooth)
    f1        = 2 * precision * recall / (precision + recall + smooth)
    dice      = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou       = (tp + smooth) / (tp + fp + fn + smooth)

    return {'iou': float(iou), 'dice': float(dice),
            'precision': float(precision), 'recall': float(recall), 'f1': float(f1)}


def evaluate_metrics(model, loader, criterion_bce, dice_loss_fn, device):
    model.eval()
    total_loss = 0.0
    all_metrics = {'iou': [], 'dice': [], 'precision': [], 'recall': [], 'f1': []}

    with torch.no_grad():
        for batch in loader:
            imgs  = batch['image'].to(device)
            masks = batch['label'].to(device)

            outputs = model(imgs)
            loss    = criterion_bce(outputs, masks) + dice_loss_fn(outputs, masks)
            total_loss += loss.item() * imgs.size(0)

            preds_np = (torch.sigmoid(outputs) > 0.5).cpu().numpy().astype(np.uint8)
            gt_np    = masks.cpu().numpy().astype(np.uint8)

            for i in range(len(preds_np)):
                m = calculate_segmentation_metrics(preds_np[i, 0], gt_np[i, 0])
                for k in all_metrics:
                    all_metrics[k].append(m[k])

    avg_loss    = total_loss / len(loader.dataset)
    avg_metrics = {k: float(np.mean(v)) for k, v in all_metrics.items()}
    return avg_loss, avg_metrics

## 4) Training loop

In [ ]:
ckpt_dir = os.path.join(proj_root, 'models', 'segmentation', 'checkpoints')
metrics_dir = os.path.join(proj_root, 'models', 'segmentation', 'metrics')
os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

epochs   = 15
best_val = float('inf')

history = {
    'train_loss': [], 'train_iou': [], 'train_dice': [],
    'train_precision': [], 'train_recall': [], 'train_f1': [],
    'val_loss': [], 'val_iou': [], 'val_dice': [],
    'val_precision': [], 'val_recall': [], 'val_f1': [],
}

for epoch in range(1, epochs + 1):
    # TRAIN
    model.train()
    train_loss = 0.0
    tp = fp = fn = 0

    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs} [Train]'):
        imgs  = batch['image'].to(device)
        masks = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion_bce(outputs, masks) + dice_loss(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        preds = torch.sigmoid(outputs) > 0.5
        tp += ((preds == 1) & (masks == 1)).sum().item()
        fp += ((preds == 1) & (masks == 0)).sum().item()
        fn += ((preds == 0) & (masks == 1)).sum().item()

    train_loss /= len(train_loader.dataset)
    train_precision = tp / (tp + fp + 1e-7)
    train_recall    = tp / (tp + fn + 1e-7)
    train_f1   = 2 * train_precision * train_recall / (train_precision + train_recall + 1e-7)
    train_iou  = tp / (tp + fp + fn + 1e-7)
    train_dice = 2 * tp / (2 * tp + fp + fn + 1e-7)

    # VALIDATION
    val_loss, val_metrics = evaluate_metrics(model, val_loader, criterion_bce, dice_loss, device)
    scheduler.step(val_loss)

    # HISTORY
    history['train_loss'].append(train_loss)
    history['train_iou'].append(train_iou)
    history['train_dice'].append(train_dice)
    history['train_precision'].append(train_precision)
    history['train_recall'].append(train_recall)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_metrics['iou'])
    history['val_dice'].append(val_metrics['dice'])
    history['val_precision'].append(val_metrics['precision'])
    history['val_recall'].append(val_metrics['recall'])
    history['val_f1'].append(val_metrics['f1'])

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"train_iou={train_iou:.4f} | val_iou={val_metrics['iou']:.4f} | "
        f"train_dice={train_dice:.4f} | val_dice={val_metrics['dice']:.4f}"
    )

    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_loss': val_loss,
            'val_metrics': val_metrics,
        }, os.path.join(ckpt_dir, 'efficientnet_unet_best.pth'))
        print(f'  ✅ Best model saved (val_loss={best_val:.4f})')

with open(os.path.join(metrics_dir, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)
print('Training history saved.')

## 5) Đánh giá trên tập test

In [ ]:
# Load best model
best_ckpt = os.path.join(ckpt_dir, 'efficientnet_unet_best.pth')
ckpt = torch.load(best_ckpt, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

test_loss, test_metrics = evaluate_metrics(model, test_loader, criterion_bce, dice_loss, device)

print('=== Test Set Metrics ===')
print(f"Loss:      {test_loss:.4f}")
print(f"IoU:       {test_metrics['iou']:.4f}")
print(f"Dice:      {test_metrics['dice']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1:        {test_metrics['f1']:.4f}")

# Lưu test metrics
test_results = {'test_loss': test_loss, **test_metrics}
with open(os.path.join(metrics_dir, 'test_metrics.json'), 'w') as f:
    json.dump(test_results, f, indent=2)
print('Test metrics saved.')

## 6) Visualization - Training curves & Sample predictions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], marker='o', label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   marker='o', label='Validation Loss')
axes[0].set_title('Training vs Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_range, history['train_iou'], marker='o', label='Train IoU')
axes[1].plot(epochs_range, history['val_iou'],   marker='o', label='Validation IoU')
axes[1].set_title('Training vs Validation IoU')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(epochs_range, history['train_dice'], marker='o', label='Train Dice')
axes[2].plot(epochs_range, history['val_dice'],   marker='o', label='Validation Dice')
axes[2].set_title('Training vs Validation Dice')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Dice')
axes[2].legend()
axes[2].grid(True)

plt.suptitle('Segmentation V1 Training History', fontsize=14)
plt.tight_layout()

viz_dir = os.path.join(proj_root, 'models', 'segmentation', 'visualizations')
os.makedirs(viz_dir, exist_ok=True)
plt.savefig(os.path.join(viz_dir, 'training_curves.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Training curves saved.')

In [ ]:
# Visualize sample predictions trên test set
model.eval()
sample_batch = next(iter(test_loader))
imgs = sample_batch['image'].to(device)
masks_gt = sample_batch['label']

with torch.no_grad():
    outputs = model(imgs)
    preds = (torch.sigmoid(outputs) > 0.5).float()

n_show = min(4, len(imgs))
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
if n_show == 1:
    axes = axes.reshape(1, -1)

# Denormalize
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

for i in range(n_show):
    img_denorm = (imgs[i].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    gt_mask = masks_gt[i, 0].numpy()
    pred_mask = preds[i, 0].cpu().numpy()

    axes[i, 0].imshow(img_denorm)
    axes[i, 0].set_title('Input Image', fontsize=9)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(gt_mask, cmap='gray')
    axes[i, 1].set_title('Ground Truth (Pseudo)', fontsize=9)
    axes[i, 1].axis('off')

    axes[i, 2].imshow(pred_mask, cmap='gray')
    axes[i, 2].set_title('Prediction', fontsize=9)
    axes[i, 2].axis('off')

plt.suptitle('Segmentation Predictions (Test Set)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(viz_dir, 'test_predictions.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Sample predictions saved.')